In [ ]:
import nltk, os, sys, random
import numpy as np

from keras.callbacks import LambdaCallback, ModelCheckpoint
from keras.models import Sequential
from keras.layers import Dense, LSTM
from keras.optimizers import RMSprop

In [ ]:
nltk.download("book")

In [ ]:
corpora_dir = "C:\\Users\\108pa\\AppData\\Roaming\\nltk_data\\corpora\\state_union"

file_list = []
for root, _, files in os.walk(corpora_dir):
  for filename in files:
    file_list.append(os.path.join(root,filename))

print(f"Read {len(file_list)} files.")

In [ ]:
docs = []
for files in file_list:
  with open(files, 'r') as fin:
    try: 
      str_form = fin.read().lower().replace('\n', '')
      docs.append(str_form)
    except UnicodeDecodeError:
      pass # some sentences have wierd characters, ignore them for now

text = ' '.join(docs)

print(f"corpus length: {len(text)}")

In [ ]:
chars = sorted(list(set(text)))
char_indices = dict((c, i) for i, c in enumerate(chars)) # character to index
indices_char = dict ((i, c) for i, c in enumerate(chars)) # index to character

In [ ]:
maxlen = 40
step = 3
sentences = []
next_chars = []
for i in range(0, len(text) - maxlen, step):
  sentences.append(text[i: i+maxlen])
  next_chars.append(text[i+maxlen])

print(f"nb sequences: {len(sentences)}")

print("Vectorization...")
# 3D Tensor that holds training data
x = np.zeros((len(sentences), maxlen, len(chars)), dtype=bool)
# 2D Tensor that holds testing data
y = np.zeros((len(sentences), len(chars)), dtype=bool)

for i, sentence in enumerate(sentences):
  for t, char in enumerate(sentence):
    # populate Tensor input
    x[i, t, char_indices[char]] = 1
  y[i, char_indices[next_chars[i]]] = 1

In [ ]:
def sample(preds, temperature=1.0):
  # helper function to sample an indesx from a probability array
  preds = np.asarray(preds).astype('float64')
  preds = np.log(preds) / temperature
  exp_preds = np.exp(preds)
  # softmax of predictions
  preds = exp_preds / np.sum(exp_preds)

  # sample a single character, with prob defined in preds
  probs = np.random.multinomial(1, preds, 1)
  
  return np.argmaax(probs)


In [ ]:
hidden_size = 128
model = Sequential()
model.add(LSTM(hidden_size, input_shape=(maxlen, len(chars))))
model.add(Dense(len(chars), activation='softmax'))
optimizer_new = RMSprop()

model.compile(loss='categorical_crossentropy', optimizer=optimizer_new)

In [ ]:
def on_epoch_end(epoch, _):
  # function to print generated text at the end of each epoch

  print(f"---- Generating text after Epoch: {epoch}")

  start_index = random.randint(0, len(text) - maxlen - 1)
  for diversity in [0.2, 0.5, 1.0, 1.2]:
    print(f"---- {diversity}")

    generated = ''
    sentence = text[start_index: start_index+maxlen]
    generated += sentence 

    print(f"---- Generating with seed: '{sentence}'")
    sys.stdout.write(generated)

    for i in range(400):
      x_pred = np.zeros((1,maxlen,len(chars)))
      for t, char in enumerate(sentence):
        x_pred[0,t,char_indices[char]] = 1.0

      preds = model.predict(x_pred, verbose=0)[0]

      next_index = sample(preds, diversity)
      next_char = indices_char[next_index]

      generated += next_char
      sentence = sentence[1:] + next_char

      sys.stdout.write(next_char)
      sys.stdout.flush()
    print()
  model.save_weights('saved_weights.hdf5', ovewrite=True)

print_callback = LambdaCallback(on_epoch_end=on_epoch_end)

# Create ModelCheckpoint callback to save model weights during training
checkpointer = ModelCheckpoint(
    filepath='checkpoint_weights.hdf5',
    monitor='loss',
    verbose=1,
    save_best_only=True,
    save_weights_only=True
)


In [ ]:
model.fit(x, y, batch_size=128, epochs=30, callbacks=[print_callback, checkpointer])